# Registering Externally hosted ML Models to OpenSearch

Prerequisit
- [model_hosting](../../model_hosting/README.md)

### Install python modules

In [ ]:
import sys
!{sys.executable} -m pip install opensearch-py

### Load helper modules

In [ ]:
import time
import pprint

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Register ML models to OpenSearch

Enable ML model registration

In [ ]:
# Hostname the OpenSearch *container* uses to reach the uvicorn model servers
# on the Docker host. Requires the container to be started with
#   --add-host=host.docker.internal:host-gateway
# (see start_opensearch.sh). Do NOT use "localhost" here -- inside the
# container that resolves to the container itself, not the host. And do NOT
# use the host's LAN IP -- it changes with the network and silently breaks
# every connector built on it.
host_ip = "host.docker.internal"

# One port per model (8000 is reserved for vLLM). Launch commands in
# model_hosting/README.md must match this map.
PORTS = {
    "e5_ml":     8001,   # intfloat/multilingual-e5-large            (dense)
    "sparse_ml": 8002,   # opensearch-neural-sparse-...-multilingual-v1 (sparse, doc-only)
    "e5_en":     8003,   # intfloat/e5-large-v2                      (dense)
    "splade_en": 8004,   # naver/splade-v3                           (sparse, symmetric)
}

In [ ]:
cluster_settings = {
    "persistent": {
        "plugins.ml_commons.only_run_on_ml_node": "false",
        "plugins.ml_commons.model_access_control_enabled": "true",
        "plugins.ml_commons.model_auto_deploy.enable": "false",
        "plugins.ml_commons.allow_registering_model_via_url": "true",
        "plugins.ml_commons.connector_access_control_enabled": "true",
        "plugins.ml_commons.connector.private_ip_enabled": "true",
        # One regex covers every encoder port (8001-8009); vLLM's 8000 stays untrusted.
        "plugins.ml_commons.trusted_connector_endpoints_regex": [
            f"^http://{host_ip}:800[1-9]/.*$"
        ]
    }
}

try:
    response = client.cluster.put_settings(body=cluster_settings)
    pprint.pprint(response)
except Exception as e:
    print(e)

Create connector

In [ ]:
def create_connector(
    client,
    name,
    endpoint,
    path,
    pre_process_function=None,
    post_process_function=None,
):
    action = {
        "action_type": "predict",
        "method": "POST",
        "url": f"http://${{parameters.endpoint}}{path}",
        "headers": {
            "Authorization": "Bearer ${credential.openAI_key}",
            "Content-Type": "application/json"
        },
        "request_body": "{ \"input\": ${parameters.input} }"
    }

    # Only needed when the connector is driven by neural-search ingest
    # processors (e.g. sparse_encoding), which supply `text_docs` instead of
    # `parameters.input`. Direct _predict callers (predict_model) omit both.
    if pre_process_function:
        action["pre_process_function"] = pre_process_function
    if post_process_function:
        action["post_process_function"] = post_process_function

    body = {
        "name": name,
        "version": 1,
        "protocol": "http",
        "parameters": {
            "endpoint": endpoint
        },

        # OpenSearch 3.5 does not like an empty credential object.
        # Your FastAPI service does not need to use this value.
        "credential": {
            "openAI_key": "dummy_value"
        },

        "actions": [action]
    }

    try:
        response = client.transport.perform_request(
            "POST",
            "/_plugins/_ml/connectors/_create",
            body=body
        )
        pprint.pprint(response)

    except Exception as e:
        print(type(e))
        print(e)

        if hasattr(e, "info"):
            pprint.pprint(e.info)

    return response["connector_id"]

In [ ]:
# One passages + one query connector per model: {model}_{lang}_{role}_connector_id
#
# Sparse models return {token: weight} maps (rank_features); dense models
# return vectors. The neural-search processors (sparse_encoding /
# text_embedding, neural_sparse / neural queries) supply `text_docs`, which
# pre_process shapes into { "input": [...] } for the sparse endpoints.
#
# Query-side semantics differ per model -- this is why every model needs BOTH
# connectors even when the URL only differs in the path:
#   sparse_ml  : doc-only encoder. /embed/passages = full MLM forward;
#                /embed/query = tokenizer + IDF only (no model forward).
#   splade_en  : SYMMETRIC SPLADE (naver/splade-v3). Both paths run the full
#                MLM forward.
#   e5_ml/e5_en: same model both paths; the server prepends the "passage: " /
#                "query: " prefix that e5 requires.

connector_ids = {}
for key, port in PORTS.items():
    sparse = key.startswith(("sparse", "splade"))
    for role in ("passages", "query"):
        connector_ids[f"{key}_{role}"] = create_connector(
            client=client,
            name=f"{key.replace('_','-')}-{role}",
            endpoint=f"{host_ip}:{port}",
            path=f"/embed/{role}",
            pre_process_function="connector.pre_process.default.embedding" if sparse else None,
        )

pprint.pprint(connector_ids)

Register a new model group

In [ ]:
def register_model_group(client, name, description="A model group for local models"):
    """
    Registers a new model group in OpenSearch ML Commons.
    
    Args:
        client: The OpenSearch client instance.
        name (str): The name of the model group.
        description (str): A description of the model group.
        
    Returns:
        str: The ID of the registered model group, or None if failed.
    """
    model_group_body = {
        "name": name,
        "description": description
    }

    try:
        response = client.transport.perform_request(
            "POST",
            "/_plugins/_ml/model_groups/_register",
            body=model_group_body
        )
        pprint.pprint(response)
        
        model_group_id = response.get("model_group_id")
        if model_group_id:
            print(f"Captured Model Group ID: {model_group_id}")
            return model_group_id
        else:
            print("Warning: Model group registered but ID not found in response.")
            return None

    except Exception as e:
        pprint.pprint(e.info)  # Print detailed error informatfion if available
        return None

In [ ]:
model_group_id = register_model_group(
    client,
    "external_encoders",
    description="Externally hosted encoder models (model_hosting/, ports 8001-8004)",
)

Register a remote model to the model group

- https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/

In [ ]:
def register_remote_model(
    client,
    model_group_id,
    connector_id,
    name,
    description=None,
):
    body = {
        "name": name,
        "function_name": "remote",
        "model_group_id": model_group_id,
        "description": description or name,
        "connector_id": connector_id,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_register",
        body=body,
    )

    pprint.pprint(response)

    return response["model_id"]

Register and deploy every model

Variable naming: `{model}_{lang}_{role}_model_id`, e.g. `splade_en_passage_model_id`.
The `*_passage_model_id`s go into the indexing notebooks' `model_id` cells; the
`*_query_model_id`s are used at search time.

In [ ]:
HF_NAMES = {
    "e5_ml":     "intfloat/multilingual-e5-large",
    "sparse_ml": "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1",
    "e5_en":     "intfloat/e5-large-v2",
    "splade_en": "naver/splade-v3",
}

model_ids = {}
for key, port in PORTS.items():
    for role in ("passages", "query"):
        register_id = register_remote_model(
            client=client,
            model_group_id=model_group_id,
            connector_id=connector_ids[f"{key}_{role}"],
            name=f"{HF_NAMES[key]} ({role})",
            description=f"{HF_NAMES[key]} {role} encoder on port {port}",
        )
        short = "passage" if role == "passages" else "query"
        model_ids[f"{key}_{short}_model_id"] = register_id

In [ ]:
def deploy_model(client, model_id, wait=True, timeout=300, poll_interval=5):
    """
    Deploy a remote model and optionally wait until deployment completes.

    Returns:
        model_id
    """

    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_deploy",
    )

    pprint.pprint(response)

    if not wait:
        return model_id

    start = time.time()

    while True:
        model_info = client.transport.perform_request(
            "GET",
            f"/_plugins/_ml/models/{model_id}",
        )

        state = model_info.get("model_state")

        print(f"model_state={state}")

        if state == "DEPLOYED":
            print(f"Model deployed: {model_id}")
            return model_id

        if state == "DEPLOY_FAILED":
            raise RuntimeError(
                f"Deployment failed:\n{pprint.pformat(model_info)}"
            )

        if time.time() - start > timeout:
            raise TimeoutError(
                f"Deployment timeout after {timeout} seconds"
            )

        time.sleep(poll_interval)

In [ ]:
for var, mid in model_ids.items():
    deploy_model(client, mid)

# Convenience variables, one per model/role
e5_ml_passage_model_id     = model_ids["e5_ml_passage_model_id"]
e5_ml_query_model_id       = model_ids["e5_ml_query_model_id"]
sparse_ml_passage_model_id = model_ids["sparse_ml_passage_model_id"]
sparse_ml_query_model_id   = model_ids["sparse_ml_query_model_id"]
e5_en_passage_model_id     = model_ids["e5_en_passage_model_id"]
e5_en_query_model_id       = model_ids["e5_en_query_model_id"]
splade_en_passage_model_id = model_ids["splade_en_passage_model_id"]
splade_en_query_model_id   = model_ids["splade_en_query_model_id"]

pprint.pprint(model_ids)

Testing the encoders

Multilingual models are tested with Japanese, English models with English.
Note the sparse query-side contrast: `sparse_ml` (doc-only) returns just the
IDF-weighted query terms, while `splade_en` (symmetric) returns an expanded
term set from a full model forward.

In [ ]:
def predict_model(client, model_id, texts):
    """
    Run inference against a deployed OpenSearch model.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        Deployed model ID.
    texts : str | list[str]
        Input text(s).

    Returns
    -------
    dict
        OpenSearch prediction response.
    """

    if isinstance(texts, str):
        texts = [texts]

    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_predict",
        body={
            "parameters": {
                "input": texts
            }
        }
    )

    return response


In [ ]:
# sparse_ml (doc-only): passage path = MLM forward, query path = tokenizer+IDF
pprint.pprint(predict_model(client, sparse_ml_passage_model_id, "情報検索と生成AIについて説明してください。"))
pprint.pprint(predict_model(client, sparse_ml_query_model_id, "情報検索と生成AIについて説明してください。"))

# splade_en (symmetric): both paths run the full MLM forward
pprint.pprint(predict_model(client, splade_en_passage_model_id, "Explain information retrieval and generative AI."))
pprint.pprint(predict_model(client, splade_en_query_model_id, "Explain information retrieval and generative AI."))

Test the dense models

In [ ]:
pprint.pprint(predict_model(client, e5_ml_passage_model_id, "情報検索と生成AIについて説明してください。"))
pprint.pprint(predict_model(client, e5_ml_query_model_id, "情報検索と生成AIについて説明してください。"))

pprint.pprint(predict_model(client, e5_en_passage_model_id, "Explain information retrieval and generative AI."))
pprint.pprint(predict_model(client, e5_en_query_model_id, "Explain information retrieval and generative AI."))

### Register and deploy the reranker

`BAAI/bge-reranker-v2-m3` (multilingual cross-encoder, port 8005) is the
second-stage reranker used over any index via an OpenSearch **search
pipeline**. Its FastAPI service exposes a Cohere-Rerank-compatible
`/v1/rerank` endpoint so the connector can use the built-in
`connector.pre_process.cohere.rerank` / `post_process.cohere.rerank`
functions -- no custom painless scripts.

In [ ]:
RERANK_PORT = 8005

reranker_connector_id = client.transport.perform_request(
    "POST", "/_plugins/_ml/connectors/_create", body={
        "name": "reranker-ml",
        "version": 1,
        "protocol": "http",
        "parameters": {"endpoint": f"{host_ip}:{RERANK_PORT}"},
        "credential": {"openAI_key": "dummy_value"},
        "actions": [{
            "action_type": "predict",
            "method": "POST",
            "url": "http://${parameters.endpoint}/v1/rerank",
            "headers": {"Authorization": "Bearer ${credential.openAI_key}",
                        "Content-Type": "application/json"},
            "request_body": "{ \"query\": \"${parameters.query}\", \"documents\": ${parameters.documents}, \"top_n\": ${parameters.top_n} }",
            "pre_process_function": "connector.pre_process.cohere.rerank",
            "post_process_function": "connector.post_process.cohere.rerank",
        }],
    })["connector_id"]

reranker_model_id = register_remote_model(
    client=client,
    model_group_id=model_group_id,
    connector_id=reranker_connector_id,
    name="BAAI/bge-reranker-v2-m3",
    description=f"Multilingual cross-encoder reranker on port {RERANK_PORT}",
)
deploy_model(client, reranker_model_id)

Create a search pipeline that reranks with it

Any query can then opt in with `?search_pipeline=rerank_bge_m3`.

In [ ]:
response = client.transport.perform_request(
    "PUT", "/_search/pipeline/rerank_bge_m3", body={
        "description": "Rerank hits with BAAI/bge-reranker-v2-m3 (remote, port 8005)",
        "response_processors": [{
            "rerank": {
                "ml_opensearch": {"model_id": reranker_model_id},
                "context": {"document_fields": ["text"]},
            }
        }],
    })
pprint.pprint(response)

Test the rerank pipeline

**Gotcha:** the query must return `_source` including the `document_fields`
(here `text`) -- with `_source` disabled the processor extracts nothing and
silently scores every hit identically (no reordering).

In [ ]:
response = client.search(
    index="msmarco_v1_passage_bm25",
    params={"search_pipeline": "rerank_bge_m3"},
    body={
        "query": {"match": {"text": "do goldfish grow"}},
        "size": 20,
        "_source": ["text"],   # REQUIRED: rerank context reads _source["text"]
        "ext": {"rerank": {"query_context": {"query_text": "do goldfish grow"}}},
    },
)
for hit in response["hits"]["hits"][:5]:
    print(round(hit["_score"], 2), hit["_source"]["text"][:100])

### List deployed models

Search ML Commons for models that are currently deployed and return their IDs.

- https://docs.opensearch.org/latest/ml-commons-plugin/api/model-apis/search-model/

In [ ]:
def list_deployed_models(
    client,
    states=("DEPLOYED", "PARTIALLY_DEPLOYED"),
    size=100,
):
    """
    List models currently deployed in OpenSearch ML Commons, grouped by model group.

    Parameters
    ----------
    client : OpenSearch client
    states : tuple[str, ...]
        Model states to include. Defaults to DEPLOYED and PARTIALLY_DEPLOYED.
    size : int
        Maximum number of models to return.

    Returns
    -------
    dict[str, list[dict]]
        Mapping of model_group_id -> list of
        {"model_id", "name", "model_state"} entries.
    """
    body = {
        "query": {
            "terms": {
                "model_state": list(states)
            }
        },
        "_source": ["name", "model_state", "model_group_id"],
        "size": size,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_search",
        body=body,
    )

    grouped = {}
    for hit in response["hits"]["hits"]:
        source = hit["_source"]
        group_id = source.get("model_group_id", "<no group>")
        grouped.setdefault(group_id, []).append(
            {
                "model_id": hit["_id"],
                "name": source.get("name"),
                "model_state": source.get("model_state"),
            }
        )

    for group_id, models in grouped.items():
        print(f"Model group: {group_id}")
        for model in models:
            print(
                f"  - {model['model_id']} "
                f"({model['name']}, {model['model_state']})"
            )
        print()

    return grouped

In [ ]:
deployed_models = list_deployed_models(client)

### Delete models

Delete a single model, or every model under a model group. A model must be
undeployed before it can be deleted, so `delete_model` undeploys first.

- https://docs.opensearch.org/latest/ml-commons-plugin/api/model-apis/delete-model/

In [ ]:
def undeploy_model(client, model_id, wait=True, timeout=300, poll_interval=5):
    """
    Undeploy a model from OpenSearch ML Commons and optionally wait until
    it is fully undeployed.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        The model to undeploy.
    wait : bool
        Poll the model state until it is UNDEPLOYED.
    timeout : int
        Maximum seconds to wait when wait=True.
    poll_interval : int
        Seconds between state checks.

    Returns
    -------
    model_id
    """
    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_undeploy",
    )

    pprint.pprint(response)

    if not wait:
        return model_id

    start = time.time()

    while True:
        model_info = client.transport.perform_request(
            "GET",
            f"/_plugins/_ml/models/{model_id}",
        )

        state = model_info.get("model_state")

        print(f"modzlwzKp8BFwfKFgbhH1bNel_state={state}")

        if state == "UNDEPLOYED":
            print(f"Model undeployed: {model_id}")
            return model_id

        if time.time() - start > timeout:
            raise TimeoutError(
                f"Undeploy timeout after {timeout} seconds"
            )

        time.sleep(poll_interval)

In [ ]:
def delete_model(client, model_id, undeploy=True):
    """
    Delete a model from OpenSearch ML Commons.

    A deployed model cannot be deleted, so it is undeployed first.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        The model to delete.
    undeploy : bool
        Undeploy the model before deleting it.

    Returns
    -------
    dict
        The delete API response.
    """
    if undeploy:
        try:
            undeploy_model(client, model_id)
        except Exception as e:
            print(f"Undeploy skipped/failed for {model_id}: {e}")

    response = client.transport.perform_request(
        "DELETE",
        f"/_plugins/_ml/models/{model_id}",
    )

    pprint.pprint(response)

    return response

In [ ]:
delete_model(client, "your-model-id-to-delete")  # Replace with the actual model ID you want to delete

In [ ]:
def delete_models_in_group(client, model_group_id, delete_group=False, size=1000):
    """
    Delete every model under a model group, and optionally the group itself.

    Parameters
    ----------
    client : OpenSearch client
    model_group_id : str
        The model group whose models should be deleted.
    delete_group : bool
        Also delete the (now empty) model group afterwards.
    size : int
        Maximum number of models to look up in the group.

    Returns
    -------
    list[str]
        The model IDs that were deleted.
    """
    body = {
        "query": {
            "term": {
                "model_group_id": model_group_id
            }
        },
        "_source": ["name", "model_state"],
        "size": size,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_search",
        body=body,
    )

    model_ids = [hit["_id"] for hit in response["hits"]["hits"]]
    print(f"Found {len(model_ids)} model(s) in group {model_group_id}")

    for model_id in model_ids:
        print(f"Deleting model {model_id}")
        delete_model(client, model_id)

    if delete_group:
        group_response = client.transport.perform_request(
            "DELETE",
            f"/_plugins/_ml/model_groups/{model_group_id}",
        )
        pprint.pprint(group_response)

    return model_ids

In [ ]:
delete_models_in_group(client, "model-group-id-to-delete", delete_group=True)  # Replace with the actual model group ID you want to delete